In [1]:
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer








In [18]:
import time
import math
from pynput import mouse

# --- VARIÁVEIS GLOBAIS ---
posicao_atual = (0, 0)
posicao_anterior = (0, 0)
distancia_acumulada = 0.0

# --- PARÂMETROS DE CALIBRAÇÃO FÍSICA ---
MOVIMENTO_NORMAL_PX = 350
SUAVIZACAO_SEGUNDOS = 3  

historico_movimento = []

# --- 1. CAPTURAR O MOUSE ---
def ao_mover(x, y):
    global posicao_atual, posicao_anterior, distancia_acumulada
    posicao_atual = (x, y)
    
    dx = posicao_atual[0] - posicao_anterior[0]
    dy = posicao_atual[1] - posicao_anterior[1]
    distancia = math.sqrt(dx**2 + dy**2)
    
    distancia_acumulada += distancia
    posicao_anterior = posicao_atual

listener = mouse.Listener(on_move=ao_mover)
listener.start()

print(f"Rastreando. Nível 3 ancorado em ~{MOVIMENTO_NORMAL_PX} px/s.")

# --- 2. LOOP PRINCIPAL ---
while True:
    time.sleep(1.0) 
    
    distancia_segundo_atual = distancia_acumulada
    distancia_acumulada = 0.0
    
    # Mantém apenas os últimos X segundos na lista
    historico_movimento.append(distancia_segundo_atual)
    if len(historico_movimento) > SUAVIZACAO_SEGUNDOS:
        historico_movimento.pop(0)
        
    # Calcula a média móvel de movimentação
    energia_media = sum(historico_movimento) / len(historico_movimento)
    
    # --- 3. MAPEAMENTO DE ESTADOS BAYESIANOS ---
    if energia_media < 10:
        nivel = 1
    elif energia_media < (MOVIMENTO_NORMAL_PX * 0.5):
        nivel = 2
    elif energia_media < (MOVIMENTO_NORMAL_PX * 1.5):
        nivel = 3
    elif energia_media < (MOVIMENTO_NORMAL_PX * 3.0):
        nivel = 4
    else:
        nivel = 5
        
    print(f"Energia Média (px/s): {energia_media:04.0f} | NÍVEL DE AGITAÇÃO: {nivel}")

Rastreando. Nível 3 ancorado em ~350 px/s.
Energia Média (px/s): 0992 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0624 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0581 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0301 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0312 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0265 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0293 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0299 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0182 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0102 | NÍVEL DE AGITAÇÃO: 2
Energia Média (px/s): 0095 | NÍVEL DE AGITAÇÃO: 2
Energia Média (px/s): 0573 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0884 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0922 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0692 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0620 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0585 | NÍVEL DE AGITAÇÃO: 4
Energia Média (px/s): 0474 | NÍVEL DE AGITAÇÃO: 3
Energia Média (px/s): 0627 | NÍVEL DE AGITAÇÃO: 4
Energia

KeyboardInterrupt: 

In [19]:
import time
import math
import json
import socket
from pynput import mouse

# --- CONFIGURAÇÃO DE COMUNICAÇÃO ---
IP_DESTINO = "127.0.0.1"
PORTA_DESTINO = 5005
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# --- VARIÁVEIS DE ESTADO E LOG ---
posicao_atual = (0, 0)
posicao_anterior = (0, 0)
distancia_acumulada = 0.0
historico_log = [] # Lista que armazenará todos os instantes medidos

# --- PARÂMETROS DE CALIBRAÇÃO ---
MOVIMENTO_NORMAL_PX = 350
SUAVIZACAO_SEGUNDOS = 3  
janela_movimento = []

def ao_mover(x, y):
    global posicao_atual, posicao_anterior, distancia_acumulada
    posicao_atual = (x, y)
    dx = posicao_atual[0] - posicao_anterior[0]
    dy = posicao_atual[1] - posicao_anterior[1]
    distancia_acumulada += math.sqrt(dx**2 + dy**2)
    posicao_anterior = posicao_atual

listener = mouse.Listener(on_move=ao_mover)
listener.start()

print(f"Exportando lista para 'historico_agitacao.json' e enviando via UDP...")

# --- LOOP PRINCIPAL ---
while True:
    time.sleep(1.0) 
    
    distancia_segundo_atual = distancia_acumulada
    distancia_acumulada = 0.0
    
    janela_movimento.append(distancia_segundo_atual)
    if len(janela_movimento) > SUAVIZACAO_SEGUNDOS:
        janela_movimento.pop(0)
        
    energia_media = sum(janela_movimento) / len(janela_movimento)
    
    # Mapeamento dos níveis
    if energia_media < 10: nivel = 1
    elif energia_media < (MOVIMENTO_NORMAL_PX * 0.5): nivel = 2
    elif energia_media < (MOVIMENTO_NORMAL_PX * 1.5): nivel = 3
    elif energia_media < (MOVIMENTO_NORMAL_PX * 3.0): nivel = 4
    else: nivel = 5
    
    # 1. CRIAR O INSTANTE ATUAL
    instante_medido = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "nivel_agitacao": nivel,
        "energia_px_s": round(energia_media, 2)
    }
    
    # 2. ADICIONAR À LISTA (O EXPORTÁVEL)
    historico_log.append(instante_medido)
    
    # 3. EXPORTAR A LISTA PARA ARQUIVO JSON
    # 'w' sobrescreve o arquivo com a lista atualizada (contendo todos os instantes)
    with open("historico_agitacao.json", "w") as f:
        json.dump(historico_log, f, indent=4)
    
    # 4. ENVIAR DADO INSTANTÂNEO VIA UDP (Para a Rede Bayesiana)
    mensagem_udp = json.dumps(instante_medido).encode('utf-8')
    sock.sendto(mensagem_udp, (IP_DESTINO, PORTA_DESTINO))
    
    print(f"Instante Medido: {instante_medido['timestamp']} | Nível: {nivel} | Lista: {len(historico_log)} itens")

Exportando lista para 'historico_agitacao.json' e enviando via UDP...
Instante Medido: 2026-03-29 02:15:40 | Nível: 5 | Lista: 1 itens
Instante Medido: 2026-03-29 02:15:41 | Nível: 4 | Lista: 2 itens
Instante Medido: 2026-03-29 02:15:42 | Nível: 4 | Lista: 3 itens
Instante Medido: 2026-03-29 02:15:43 | Nível: 3 | Lista: 4 itens
Instante Medido: 2026-03-29 02:15:44 | Nível: 4 | Lista: 5 itens
Instante Medido: 2026-03-29 02:15:45 | Nível: 3 | Lista: 6 itens
Instante Medido: 2026-03-29 02:15:46 | Nível: 3 | Lista: 7 itens
Instante Medido: 2026-03-29 02:15:47 | Nível: 2 | Lista: 8 itens
Instante Medido: 2026-03-29 02:15:48 | Nível: 3 | Lista: 9 itens
Instante Medido: 2026-03-29 02:15:49 | Nível: 3 | Lista: 10 itens
Instante Medido: 2026-03-29 02:15:50 | Nível: 4 | Lista: 11 itens
Instante Medido: 2026-03-29 02:15:51 | Nível: 4 | Lista: 12 itens
Instante Medido: 2026-03-29 02:15:52 | Nível: 4 | Lista: 13 itens
Instante Medido: 2026-03-29 02:15:53 | Nível: 4 | Lista: 14 itens
Instante Medido

KeyboardInterrupt: 